# Notebook 01 — 데이터셋 준비 & VQA 포맷 변환

## 목표
NEU Metal Surface Defects 데이터셋을 로드하고  
Qwen2.5-VL 파인튜닝용 **VQA(Visual Question Answering) 포맷**으로 변환한다.

## 파이프라인
```
NEU 원본 (이미지 + 클래스 레이블)
        ↓
EDA — 클래스 분포, 이미지 샘플 시각화
        ↓
Instruction 템플릿 생성
{"role": "user", "content": [image, "이 이미지의 불량 유형을 JSON으로 분석해줘"]}
{"role": "assistant", "content": '{"type": "scratches", "severity": "high", ...}'}
        ↓
Train / Val / Test 분할 (70 / 15 / 15)
        ↓
data/processed/train.json, val.json, test.json 저장
```

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os, json, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import Counter

ROOT = Path("..") 
DATA_RAW = ROOT / "data" / "raw" / "NEU-DET"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print("경로 확인:")
print(f"  raw     : {DATA_RAW.resolve()}")
print(f"  processed: {DATA_PROCESSED.resolve()}")

## 1. 데이터셋 다운로드

NEU Metal Surface Defects Dataset  
- 출처: Northeastern University (공개 데이터셋)  
- 클래스 6종 × 300장 = **1,800장**  
- 이미지 크기: 200×200 픽셀, 그레이스케일

In [ ]:
import subprocess, sys

# kaggle CLI로 다운로드 (없으면 pip install kaggle 후 ~/.kaggle/kaggle.json 설정)
# 또는 HuggingFace datasets로 대체
try:
    import kaggle
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "kaggle", "datasets", "download",
        "-d", "kaustubhdikshit/neu-surface-defect-database",
        "-p", str(DATA_RAW), "--unzip"
    ], check=True)
    print("Kaggle 다운로드 완료")
except Exception as e:
    print(f"Kaggle 다운로드 실패: {e}")
    print("대안: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database")
    print("      위 링크에서 수동 다운로드 후 data/raw/NEU-DET/ 에 압축 해제")

# 폴더 구조 확인
if DATA_RAW.exists():
    for p in sorted(DATA_RAW.iterdir()):
        if p.is_dir():
            count = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.bmp")))
            print(f"  {p.name:25s}: {count}장")
else:
    print(f"⚠ {DATA_RAW} 폴더 없음 — 수동 다운로드 필요")

## 2. EDA — 탐색적 데이터 분석

In [ ]:
# 6가지 불량 클래스 정의
DEFECT_CLASSES = {
    "crazing":         {"ko": "균열",     "severity_rule": lambda name: "low"},
    "inclusion":       {"ko": "개재물",   "severity_rule": lambda name: "medium"},
    "patches":         {"ko": "패치결함", "severity_rule": lambda name: "low"},
    "pitted_surface":  {"ko": "피팅",     "severity_rule": lambda name: "high"},
    "rolled-in_scale": {"ko": "압연스케일","severity_rule": lambda name: "medium"},
    "scratches":       {"ko": "스크래치", "severity_rule": lambda name: "high"},
}

# 데이터 수집
records = []

for cls_name, meta in DEFECT_CLASSES.items():
    # NEU 데이터셋 폴더 구조에 따라 경로 탐색
    # 가능한 경로 패턴 시도
    possible_dirs = [
        DATA_RAW / cls_name,
        DATA_RAW / "NEU-DET" / cls_name,
        DATA_RAW / "train" / cls_name,
    ]
    cls_dir = None
    for d in possible_dirs:
        if d.exists():
            cls_dir = d
            break

    if cls_dir is None:
        print(f"⚠ '{cls_name}' 폴더를 찾을 수 없음")
        continue

    for img_path in sorted(cls_dir.glob("*")):
        if img_path.suffix.lower() in (".jpg", ".jpeg", ".bmp", ".png"):
            records.append({
                "path": str(img_path),
                "class": cls_name,
                "class_ko": meta["ko"],
                "severity": meta["severity_rule"](img_path.name),
            })

df = pd.DataFrame(records)
print(f"총 이미지 수: {len(df)}")
print(df["class"].value_counts())

In [ ]:
# 클래스 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 클래스별 이미지 수
counts = df["class"].value_counts()
colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title("클래스별 이미지 수", fontsize=13)
axes[0].set_xlabel("불량 유형")
axes[0].set_ylabel("이미지 수")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=10)

# severity 분포
sev_counts = df["severity"].value_counts()
axes[1].pie(sev_counts.values, labels=sev_counts.index, autopct='%1.1f%%',
            colors=['#90EE90', '#FFD700', '#FF6B6B'], startangle=90)
axes[1].set_title("심각도 분포", fontsize=13)

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "eda_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장: data/processed/eda_distribution.png")

In [ ]:
# 각 클래스별 샘플 이미지 시각화
n_cols = 6
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 9))

for col_idx, cls_name in enumerate(DEFECT_CLASSES.keys()):
    cls_df = df[df["class"] == cls_name].sample(n=min(n_rows, len(df[df["class"]==cls_name])), random_state=42)
    for row_idx, (_, row) in enumerate(cls_df.iterrows()):
        ax = axes[row_idx][col_idx]
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(f"{cls_name}\n({DEFECT_CLASSES[cls_name]['ko']})" if row_idx == 0 else "", fontsize=9)
        ax.axis("off")

plt.suptitle("NEU 불량 유형별 샘플 이미지 (클래스별 3장)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / "eda_samples.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장: data/processed/eda_samples.png")

## 3. VQA Instruction 포맷 변환

Qwen2.5-VL은 Chat 형식의 멀티모달 instruction을 받는다.  
각 이미지를 **질문-답변 쌍**으로 변환한다.

```
{
  "id": "crazing_001",
  "image": "data/raw/NEU-DET/crazing/crazing_1.jpg",
  "conversations": [
    {
      "role": "user",
      "content": "<image>\n이 금속 표면 이미지를 분석하고 불량 정보를 JSON 형식으로 출력해줘."
    },
    {
      "role": "assistant", 
      "content": "{\"type\": \"crazing\", \"type_ko\": \"균열\", \"severity\": \"low\", \"description\": \"금속 표면에 미세한 균열 패턴이 관찰됩니다. 표면 전반에 걸쳐 망상형 크랙이 분포합니다.\"}"
    }
  ]
}
```

In [ ]:
# 클래스별 설명 템플릿 (다양성 위해 3가지 변형)
DESCRIPTION_TEMPLATES = {
    "crazing": [
        "금속 표면에 미세한 균열 패턴이 관찰됩니다. 표면 전반에 걸쳐 망상형 크랙이 분포합니다.",
        "표면 균열(crazing)이 감지되었습니다. 불규칙한 망상 패턴의 미세 크랙이 나타납니다.",
        "표면에 균열성 결함이 존재합니다. 열응력 또는 냉각 불균일로 인한 망상 균열 패턴입니다.",
    ],
    "inclusion": [
        "금속 내부 이물질(개재물)이 표면에 노출되어 있습니다. 어두운 점 또는 반점 형태로 나타납니다.",
        "개재물 결함이 탐지되었습니다. 비금속 이물질이 롤링 과정에서 표면에 삽입된 형태입니다.",
        "표면에 포함물(inclusion) 결함이 있습니다. 소재 내부의 불순물이 표면에 노출된 상태입니다.",
    ],
    "patches": [
        "표면에 패치 형태의 결함이 관찰됩니다. 질감이 다른 영역이 불규칙하게 분포합니다.",
        "패치 결함이 감지되었습니다. 표면 코팅 불균일 또는 산화로 인한 불규칙 영역입니다.",
        "표면에 얼룩 또는 패치 패턴의 결함이 있습니다. 색조 및 질감 차이가 뚜렷합니다.",
    ],
    "pitted_surface": [
        "표면에 피팅(pitting) 결함이 발견되었습니다. 다수의 소형 구멍 또는 함몰부가 분포합니다.",
        "피팅 부식으로 인한 표면 손상이 관찰됩니다. 점상 결함이 표면 전반에 걸쳐 나타납니다.",
        "표면 피팅 결함입니다. 전기화학적 부식 또는 기계적 충격에 의한 소형 크레이터가 분포합니다.",
    ],
    "rolled-in_scale": [
        "압연 과정에서 스케일(산화막)이 표면에 삽입된 결함입니다. 선형 또는 층상 패턴이 특징입니다.",
        "롤링 스케일 결함이 탐지되었습니다. 고온 압연 시 산화물이 표면에 압착된 형태입니다.",
        "압연 스케일 결함이 관찰됩니다. 표면에 불규칙한 층상 이물질이 삽입되어 있습니다.",
    ],
    "scratches": [
        "표면에 선형 스크래치 결함이 관찰됩니다. 방향성 있는 긁힘 자국이 선명하게 나타납니다.",
        "스크래치 결함이 감지되었습니다. 기계적 마찰에 의한 표면 손상으로 방향성이 있습니다.",
        "표면 긁힘(scratches) 결함입니다. 선명한 선형 패턴이 특정 방향으로 분포합니다.",
    ],
}

# 질문 프롬프트 변형 (다양성)
QUESTION_TEMPLATES = [
    "<image>\n이 금속 표면 이미지를 분석하고 불량 정보를 JSON 형식으로 출력해줘.\n출력 형식: {\"type\": \"...\", \"type_ko\": \"...\", \"severity\": \"low|medium|high\", \"description\": \"...\"}",
    "<image>\n금속 표면의 불량 유형을 파악하고 다음 JSON 구조로 답변해줘:\n{\"type\": \"불량유형\", \"type_ko\": \"한글명\", \"severity\": \"심각도\", \"description\": \"설명\"}",
    "<image>\n이 이미지는 금속 제품 표면입니다. 불량 유형과 심각도를 분석해서 JSON으로 출력해줘.",
]

print("프롬프트 템플릿 준비 완료")
print(f"  질문 변형: {len(QUESTION_TEMPLATES)}개")
print(f"  설명 변형: {max(len(v) for v in DESCRIPTION_TEMPLATES.values())}개 (클래스당)")

In [ ]:
def build_vqa_record(idx: int, row: pd.Series) -> dict:
    cls = row["class"]
    q_template = QUESTION_TEMPLATES[idx % len(QUESTION_TEMPLATES)]
    desc = DESCRIPTION_TEMPLATES[cls][idx % len(DESCRIPTION_TEMPLATES[cls])]
    answer = json.dumps({
        "type": cls,
        "type_ko": DEFECT_CLASSES[cls]["ko"],
        "severity": row["severity"],
        "description": desc,
    }, ensure_ascii=False)
    return {
        "id": f"{cls}_{idx:04d}",
        "image": row["path"],
        "conversations": [
            {"role": "user",      "content": q_template},
            {"role": "assistant", "content": answer},
        ],
    }

# 전체 VQA 레코드 생성
vqa_records = [
    build_vqa_record(i, row)
    for i, (_, row) in enumerate(df.iterrows())
]

# 샘플 확인
print(f"VQA 레코드 총 {len(vqa_records)}개 생성")
print("\n=== 샘플 레코드 ===")
sample = vqa_records[0]
print(f"ID: {sample['id']}")
print(f"Image: {sample['image']}")
print(f"Q: {sample['conversations'][0]['content'][:80]}...")
print(f"A: {sample['conversations'][1]['content']}")

## 4. Train / Val / Test 분할 및 저장

In [ ]:
from sklearn.model_selection import train_test_split

# 클래스별 균등 분할 (stratified)
labels = [r["id"].rsplit("_", 1)[0] for r in vqa_records]

train_val, test = train_test_split(vqa_records, test_size=0.15, random_state=42, stratify=labels)
labels_tv = [r["id"].rsplit("_", 1)[0] for r in train_val]
train, val = train_test_split(train_val, test_size=0.176, random_state=42, stratify=labels_tv)
# 0.176 ≈ 15/85 → 전체 기준 약 15%

print(f"Train: {len(train):4d}장  ({len(train)/len(vqa_records)*100:.1f}%)")
print(f"Val  : {len(val):4d}장  ({len(val)/len(vqa_records)*100:.1f}%)")
print(f"Test : {len(test):4d}장  ({len(test)/len(vqa_records)*100:.1f}%)")
print(f"합계 : {len(train)+len(val)+len(test)}장")

# JSON 저장
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    out_path = DATA_PROCESSED / f"{split_name}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"저장: {out_path}")

# 분할 통계 시각화
fig, ax = plt.subplots(figsize=(8, 4))
splits = {"Train": len(train), "Val": len(val), "Test": len(test)}
bars = ax.bar(splits.keys(), splits.values(), color=["#4C72B0", "#55A868", "#C44E52"])
for bar, val_ in zip(bars, splits.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val_), ha='center', fontsize=12)
ax.set_title("데이터 분할 결과", fontsize=13)
ax.set_ylabel("이미지 수")
plt.tight_layout()
plt.show()

In [ ]:
## 5. 완료 요약

print("=" * 50)
print("Notebook 01 완료")
print("=" * 50)
print(f"데이터셋 : NEU Metal Surface Defects")
print(f"클래스   : {list(DEFECT_CLASSES.keys())}")
print(f"총 이미지: {len(vqa_records)}장")
print(f"Train    : {len(train)}장")
print(f"Val      : {len(val)}장")
print(f"Test     : {len(test)}장")
print()
print("생성된 파일:")
print("  data/processed/train.json")
print("  data/processed/val.json")
print("  data/processed/test.json")
print("  data/processed/eda_distribution.png")
print("  data/processed/eda_samples.png")
print()
print("다음 단계: 02_baseline.ipynb")
print("  → Qwen2.5-VL Zero-shot 평가 (파인튜닝 전 베이스라인)")